# 0. import libraries

In [1]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F
import gc

In [3]:
from huggingface_hub import hf_hub_download

WIDTH = "16k"
L0 = "small"

transcoder_paths = {}
for layer in range(34):
    path = hf_hub_download(
        repo_id="google/gemma-scope-2-4b-pt",
        filename=f"transcoder_all/layer_{layer}_width_{WIDTH}_l0_{L0}/params.safetensors"
    )
    transcoder_paths[layer] = path

In [4]:
from circuit_tracer.transcoder.single_layer_transcoder import load_transcoder_set
import torch

transcoder_set = load_transcoder_set(
    transcoder_paths=transcoder_paths,
    scan="gemma-scope-2-4b-pt",
    feature_input_hook="hook_resid_mid",
    feature_output_hook="hook_mlp_out",
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    lazy_encoder=False,
    lazy_decoder=True,
    special_load_fn="gemma-scope-2",
)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/circuit_tracer/transcoder/single_layer_transcoder.py:513: UserWarning: Lazy loading is not supported for GemmaScope2 format due to different key naming conventions. Setting lazy_encoder=False and lazy_decoder=False.
  warnings.warn(


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-4b-pt")

model = ReplacementModel.from_pretrained_and_transcoders(
    model_name="google/gemma-3-4b-pt",
    transcoders=transcoder_set,
    backend="transformerlens",
    dtype=torch.bfloat16,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-3-4b-pt into HookedTransformer


# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [6]:
max_n_logits = 5  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 1028  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 8  # Batch size when attributing
offload = "disk"
verbose = True  # Whether to display a tqdm progress bar and timing report

In [7]:
base_prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,
        batch_size=batch_size,
        offload=offload,
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files/gemma-3-4b",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

    del graph
    gc.collect()
    torch.cuda.empty_cache()

Phase 0: Precomputing activations and vectors


Generated: He ate it and then he had to crap it.


Precomputation completed in 0.29s
Found 2602750 active features
Phase 1: Running forward pass
Forward pass completed in 0.08s
Phase 2: Building input vectors
Using 5 salient logits with cumulative probability 0.3867
Will include 1028 of 2602750 feature nodes
Input vectors built in 3.81s
Phase 3: Computing logit attributions
sys:1: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
Logit attributions completed in 0.25s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1028/1028 [00:38<00:00, 26.38it/s]
Feature attributions completed in 38.97s
Attribution completed in 57.47s


✓ step 00 → 'He'  saved as 'step-00-He'


Phase 0: Precomputing activations and vectors
Precomputation completed in 0.24s
Found 2761914 active features
Phase 1: Running forward pass
Forward pass completed in 0.07s
Phase 2: Building input vectors
Using 5 salient logits with cumulative probability 0.2754
Will include 1028 of 2761914 feature nodes
Input vectors built in 3.82s
Phase 3: Computing logit attributions
Logit attributions completed in 0.24s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1028/1028 [00:39<00:00, 25.93it/s]
Feature attributions completed in 39.65s
Attribution completed in 55.79s


...

✓ step 08 → ' crap'  saved as 'step-08-crap'


Phase 0: Precomputing activations and vectors
Precomputation completed in 0.32s
Found 3981456 active features
Phase 1: Running forward pass
Forward pass completed in 0.08s
Phase 2: Building input vectors
Using 1 salient logits with cumulative probability 0.9766
Will include 1028 of 3981456 feature nodes
Input vectors built in 4.05s
Phase 3: Computing logit attributions
Logit attributions completed in 0.33s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1028/1028 [00:56<00:00, 18.16it/s]
Feature attributions completed in 56.60s
Attribution completed in 73.03s


✓ step 09 → ' it'  saved as 'step-09-it'


Phase 0: Precomputing activations and vectors
Precomputation completed in 0.38s
Found 4132754 active features
Phase 1: Running forward pass
Forward pass completed in 0.08s
Phase 2: Building input vectors
Using 5 salient logits with cumulative probability 0.9531
Will include 1028 of 4132754 feature nodes
Input vectors built in 3.97s
Phase 3: Computing logit attributions
Logit attributions completed in 0.48s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 1028/1028 [00:57<00:00, 17.96it/s]
Feature attributions completed in 57.24s
Attribution completed in 73.59s


✓ step 10 → '.'  saved as 'step-10-.'


In [8]:
# Run this to see what's actually on GPU
print(torch.cuda.memory_summary(abbreviated=True))

# Also check:
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 3         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  14294 MiB |  95256 MiB | 185006 GiB | 184992 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  14294 MiB |  95256 MiB | 185006 GiB | 184992 GiB |
|---------------------------------------------------------------------------|
| Requested memory      |  14294 MiB |  95256 MiB | 185006 GiB | 184992 GiB |
|---------------------------------------------------------------

# 2. attribute graph visualization

In [9]:
import json
from pathlib import Path

graph_dir = Path("./graph_files/gemma-3-4b")

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"gemma-3-4b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

step-00-He.json: 14 layers → ['gemma-3-4b/20-gemmascope-transcoder-16k', 'gemma-3-4b/21-gemmascope-transcoder-16k', 'gemma-3-4b/22-gemmascope-transcoder-16k']...
step-01-ate.json: 15 layers → ['gemma-3-4b/17-gemmascope-transcoder-16k', 'gemma-3-4b/20-gemmascope-transcoder-16k', 'gemma-3-4b/21-gemmascope-transcoder-16k']...
step-02-it.json: 16 layers → ['gemma-3-4b/14-gemmascope-transcoder-16k', 'gemma-3-4b/17-gemmascope-transcoder-16k', 'gemma-3-4b/20-gemmascope-transcoder-16k']...
step-03-and.json: 17 layers → ['gemma-3-4b/14-gemmascope-transcoder-16k', 'gemma-3-4b/17-gemmascope-transcoder-16k', 'gemma-3-4b/18-gemmascope-transcoder-16k']...
step-04-then.json: 16 layers → ['gemma-3-4b/14-gemmascope-transcoder-16k', 'gemma-3-4b/17-gemmascope-transcoder-16k', 'gemma-3-4b/20-gemmascope-transcoder-16k']...
step-05-he.json: 15 layers → ['gemma-3-4b/17-gemmascope-transcoder-16k', 'gemma-3-4b/20-gemmascope-transcoder-16k', 'gemma-3-4b/21-gemmascope-transcoder-16k']...
step-06-had.json: 15 lay

In [10]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(Path("./graph_files/gemma-3-4b").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/gemma-3-4b/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [ ]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/gemma-3-4b/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

## 2.1 feature analysis based on circuit

In [11]:
# inspect one graph file
data = json.loads(list(Path("./graph_files/gemma-3-4b/").glob("step-*.json"))[0].read_text())
for n in data['nodes'][:10]:
    print(n)

{'node_id': '17_1886_19', 'feature': 1813542, 'layer': '17', 'ctx_idx': 19, 'feature_type': 'cross layer transcoder', 'token_prob': 0.0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '17_1886-0', 'clerp': '', 'influence': 0.626166820526123, 'activation': 282624.0}
{'node_id': '20_170_19', 'feature': 18315, 'layer': '20', 'ctx_idx': 19, 'feature_type': 'cross layer transcoder', 'token_prob': 0.0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '20_170-0', 'clerp': '', 'influence': 0.7622352242469788, 'activation': 495616.0}
{'node_id': '20_345_19', 'feature': 67140, 'layer': '20', 'ctx_idx': 19, 'feature_type': 'cross layer transcoder', 'token_prob': 0.0, 'is_target_logit': False, 'run_idx': 0, 'reverse_ctx_idx': 0, 'jsNodeId': '20_345-0', 'clerp': '', 'influence': 0.5322176218032837, 'activation': 401408.0}
{'node_id': '21_875_19', 'feature': 402731, 'layer': '21', 'ctx_idx': 19, 'feature_type': 'cross layer transcoder', 'token_prob

In [12]:
import json
from pathlib import Path

top_features = set()

for json_path in sorted(Path("./graph_files/gemma-3-4b/").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    nodes = [n for n in data['nodes'] 
             if not n['is_target_logit'] 
             and n['influence'] is not None
             and n['feature_type'] == 'cross layer transcoder']  # ← only transcoder nodes
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:10]
    for node in nodes:
        parts = node['node_id'].split('_')
        layer = parts[0]   # '13', '14', '15' etc
        feat = parts[1]    # '1005', '2100' etc
        
        if layer.isdigit():
            top_features.add((int(layer), int(feat)))

In [13]:
print(f"Unique top features: {len(top_features)}")
print(list(top_features)[:5])

Unique top features: 106
[(29, 9072), (28, 6117), (28, 14156), (33, 701), (25, 1609)]


In [14]:
from collections import Counter

layer_counts = Counter(layer for layer, feat in top_features)
for layer in sorted(layer_counts):
    print(f"Layer {layer}: {layer_counts[layer]} features")

Layer 21: 1 features
Layer 22: 2 features
Layer 24: 1 features
Layer 25: 5 features
Layer 26: 6 features
Layer 27: 7 features
Layer 28: 24 features
Layer 29: 16 features
Layer 30: 8 features
Layer 31: 7 features
Layer 32: 19 features
Layer 33: 10 features


# 3. intervention
based on intervention_demo

In [15]:
from collections import namedtuple
from functools import partial

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

In [16]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [ ]:
supernode_features = [
    #Feature(layer=17,pos=-1,feature_idx=15160),
    #Feature(layer=17,pos=-1,feature_idx=5452),
    Feature(layer=16,pos=-1,feature_idx=11787),
    #Feature(layer=16,pos=-1,feature_idx=5798),
    #Feature(layer=17,pos=-1,feature_idx=15724),
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [ ]:
s = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"

In [ ]:
with torch.inference_mode():
    original_logits, _  = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

In [ ]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

# suppress both candidate features at the newline position (-1)
intervention_tuples = [
    (5, -1, 6651, 0.0),
    (17, -1, 15160, 0.0),
    (17, -1, 5452, 0.0),
    (16, -1, 11787, 0.0),
    (16, -1, 5798, 0.0),
    (17, -1, 15724, 0.0),
]

print("Generating with original features...")
pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=True)[0]]

print("\nGenerating with interventions...")
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=True)[0]]

display_generations_comparison(s, pre, post)